# Interactive Dashboards &mdash; `nb.dashboard()`

`UnichartNotebook` plotting methods each return a Plotly figure. The companion module
`unichart_dashboard` wires several of them into a single **interactive Dash board** that
renders inline in this notebook.

Each panel is independent and carries its own controls:

| Control | Effect |
|---|---|
| **plot type** | switch the panel between `plot`, `plot_ymult`, `bar`, `box`, `histogram` |
| **x** | choose the x column |
| **y** | choose one or more y columns (multi-select) |
| **datasets** | toggle which sets are shown (per panel) |
| **suptitle** | edit the panel title |
| **legend** | `above` / `right` / `off` (applies to `plot` / `plot_ymult`) |

Changing any control re-plots **only that panel**, reusing all of the notebook's
formatting (axis limits, variable formats, lines, highlights).

> Requires Dash (`pip install dash`). It is an optional dependency &mdash; the rest of the
> toolkit imports fine without it.

## 0. Setup &mdash; two sensor-log datasets

Two runs of a rig logged over a `time` sweep, with continuous channels (`pressure`,
`temp`, `flow`) and a categorical `phase`. The mix of continuous and categorical columns
lets the same board show line plots, bars, boxes, and histograms.

In [ ]:
# --- make repo-root importable (notebook lives in demo_notebooks/) ---
import sys, os
_repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import numpy as np
import pandas as pd
from unichart import UnichartNotebook

rng = np.random.default_rng(0)

def run(scale):
    t = np.arange(0, 60)
    phase = np.where(t < 15, 'idle', np.where(t < 40, 'ramp', 'peak'))
    return pd.DataFrame({
        'time':     t,
        'pressure': scale * (10 + 0.8 * t + rng.normal(0, 1.5, t.size)),
        'temp':     scale * (300 + 6 * t + rng.normal(0, 8, t.size)),
        'flow':     scale * (5 + 0.3 * t + rng.normal(0, 0.5, t.size)),
        'phase':    phase,
    })

uc = UnichartNotebook()
uc.load(run(1.00), title='Run A')   # Set 0
uc.load(run(1.15), title='Run B')   # Set 1
uc.sets[0].df.head()

## 1. Minimal board

Pass the notebook and a list of **panel specs**. Each spec seeds that panel's initial
controls; `method` defaults to `'plot'`. The call launches a Dash server and renders it
inline (`jupyter_mode='inline'`).

Try it: change the **plot type**, **x/y**, or untick a **dataset** on either panel and
watch just that panel update.

In [ ]:
uc.dashboard([
    {'method': 'plot', 'x': 'time', 'y': 'pressure'},
    {'method': 'plot', 'x': 'time', 'y': 'temp'},
])

## 2. A richer board

Panels can mix plot types and seed any control. `y` accepts a list (multi-select; useful
for `plot_ymult`). `datasets` seeds which sets are selected for that panel &mdash; here the
box panel starts with only Run A.

`ncols` sets the grid width and `height` the px height of each panel's graph.

In [ ]:
uc.dashboard(
    panels=[
        {'method': 'plot',      'x': 'time',  'y': ['pressure', 'temp'], 'suptitle': 'Channels vs time'},
        {'method': 'plot_ymult','x': 'time',  'y': ['pressure', 'flow'], 'legend': 'right'},
        {'method': 'bar',       'x': 'phase', 'y': 'flow',  'suptitle': 'Mean flow by phase'},
        {'method': 'box',       'x': 'phase', 'y': 'temp',  'datasets': [0]},
        {'method': 'histogram', 'x': 'temp',  'y': 'temp'},
    ],
    ncols=2,
    height=380,
)

## 3. Options & notes

- **Run target.** `jupyter_mode='inline'` (default) renders in the cell. Use
  `jupyter_mode='external'` (or `'tab'`) to open it in a browser instead, e.g.
  `uc.dashboard(panels, jupyter_mode='external', port=8051)`.
- **Plot types.** The live switch covers `plot`, `plot_ymult`, `bar`, `box`, `histogram`.
  `contour` is excluded because it needs a `z` column.
- **`legend`** only applies to `plot` / `plot_ymult`; it is ignored for the other types.
- **Shared state.** Dataset on/off toggles flip each set's `.select` just for that panel's
  render and restore it afterward, so panels stay independent. Renders are serialized with
  a lock, so keep the board to a single user / kernel.
- **Errors** in a panel (e.g. an empty `y`) show as a message inside that panel rather than
  crashing the whole board.

- **Export data.** Each panel has a **⬇ data** button that downloads its plotted
  data as CSV (tidy `trace, x, y[, z]`; contour exports the surface grid plus
  overlay points). The toolbar's **⬇ export all panels** button combines every
  panel into one CSV with a `panel` column.

### Under the hood

`uc.dashboard(...)` is a thin wrapper around `unichart_dashboard.dashboard(uc, panels)`.
For testing or custom embedding you can build the app without launching a server with
`unichart_dashboard.build_app(uc, panels)`, or render a single panel to a figure with
`unichart_dashboard.render_panel(uc, 'plot', 'time', ['pressure'], dataset_indices=[0, 1])`.